# DCLens — Quick Start Guide

This notebook walks from a lens catalogue through buffer probing, GPU map
simulation, visualization, finite-source light-curve extraction and disk reload.
It also includes sequence and optional analysis examples, followed by the complete
parameter reference for every public module-defined callable.

**Requirements:** install the package with `python -m pip install -e '.[notebook]'`
from the repository root, then choose that Python environment as the notebook
kernel. CUDA/NVIDIA drivers and OpenGL/EGL are required for simulation; see the
README installation section. Install `.[analysis]` for the optional analysis cells.

Cells are deliberately unexecuted. Running them performs real GPU work and writes
example data/results under a newly created directory in the current working
directory. Small illustrative settings below override the larger library defaults;
choose production resolution and resource limits for your scientific problem.
Run the main sections in order. Sequence and analysis sections are opt-in.


In [ ]:
%matplotlib inline
from pathlib import Path
from tempfile import mkdtemp
import numpy as np
import matplotlib.pyplot as plt

from DCLens import (
    Lens, SimulationConfig, BufferSizeProbe, MagMapSim, PMBuilder,
    MagnificationMap, MagnificationMapSeries, MagMapShow,
    IALightCurveExtractor, LightCurveResult,
)
from DCLens.physics import split_into_planes, apply_p3m_grid

run_dir = Path(mkdtemp(prefix='dclens_quick_start_', dir='.')).resolve()
print('Example data and results:', run_dir)


## 1. Data and Lens

A snapshot has one header and seven whitespace-separated columns. Masses use
solar masses, x/y/z use pc, and velocities are carried as input only. File z is
relative to `D_L_pc`; `load_frame()` adds that distance and discards absolute z ≤ 0.
This two-lens catalogue is an illustrative input, not a performance benchmark.

All `Lens` parameters are explicit below. `pattern` is only used for a directory;
a single path selects single-map mode. The loader never integrates velocities.


In [ ]:
catalogue = np.array([
    [1.0, -0.0001, 0.0, 0.0, 0.0, 0.0, 0.0],
    [1.0,  0.0001, 0.0, 0.0, 0.0, 0.0, 0.0],
])
lens_file = run_dir / 'lens.dat'
np.savetxt(lens_file, catalogue,
           header='M_Msun x_pc y_pc z_pc vx vy vz', comments='')
lens = Lens(source=lens_file, D_L_pc=1e4, D_S_pc=1e9,
            N_planes=1, pattern='*.dat')
table = lens.load_frame(i=0)
print(lens.mode, lens.frame_count, lens.files)
display(table)
# Lazy iteration is available for both a single file and a sequence.
for frame_table in lens.iter_frames():
    print('Lenses in frame:', len(frame_table))


## 2. Settings (every configuration field)

All 17 configuration arguments are shown. The full reference at the end lists
library defaults, units and constraints. Here `max_active_cells=200_000` uses
about 33.6 MiB for the three IPM queues, before other allocations; the library
**default is 20,000,000** slots (about 3.28 GiB). These examples are not a guarantee
that an arbitrary simulation fits available GPU memory.

Use `deflector='p3m'` for PM + local PP correction, or `'pp'` for direct all-lens
summation. Use the same choice for probing and simulation. Config is immutable;
per-call keyword overrides leave it unchanged.


In [ ]:
config = SimulationConfig(
    coarse_ps=0.002,
    finest_ps=0.0005,
    distortion_threshold=0.2,
    PM_grid=2048,
    img_center=(0.0, 0.0),
    src_considered_region=(0.04, 0.04),
    buffer_size=0.1,
    max_refine_iters=20,
    srcp_ps=0.0005,
    ss=2,
    max_coarse_per_chunk=20,
    max_active_cells=200_000,
    p3m_a_pm_over_H=6.0,
    p3m_padding_factor=2.0,
    show_progress=True,
    verbose=False,
    deflector='p3m',
)


## 3. PMBuilder (optional host inspection)

Simulation performs this preparation automatically. This cell demonstrates the
standalone `split_into_planes` and `PMBuilder` interfaces with all parameters.
`PMBuilder` is the same function as `apply_p3m_grid`; it modifies the supplied
plane list, reorders lenses by bucket, and returns that list. It prepares host
layout only. GPU mass assignment/FFT happens later during engine upload.

P3MDeflector, IPM, ASR and Rasterization Shader are internal native stages invoked
by `MagMapSim`; there are no separate public Python constructors for them.


In [ ]:
planes = split_into_planes(BH_table=table, N_planes=lens.N_planes)
prepared = PMBuilder(
    lens_planes=planes, grid_nx=config.PM_grid, grid_ny=config.PM_grid,
    theta_bounds_arcsec=(-0.12, 0.12, -0.12, 0.12),
    a_pm_over_H=config.p3m_a_pm_over_H,
    padding_factor=config.p3m_padding_factor,
)
print([{key: p['p3m'][key] for key in ('pm_nx', 'pm_ny', 'H', 'a_pm')}
       for p in prepared])
# Equivalent legacy callable (use a fresh plane list because the builder mutates):
# prepared = apply_p3m_grid(lens_planes=split_into_planes(table, lens.N_planes),
#     grid_nx=128, grid_ny=128, theta_bounds_arcsec=(-0.12, 0.12, -0.12, 0.12),
#     a_pm_over_H=6.0, padding_factor=2.0)
del prepared, planes


## 4. BufferSizeProbe

All named parameters are shown. `**keywords` accepts any of the 17 config fields
already shown above; there is no second hidden set of simulation parameters.
`return_frames=True` gives both the maximum recommended margin and each snapshot's
margin. `False` returns just the scalar. The probe is an eight-point heuristic.
Reaching the maximum search range should prompt a larger range before simulation;
the probe does not change `config.buffer_size` automatically.


In [ ]:
probe = BufferSizeProbe(
    lens=lens, config=config, buffer_step=0.01,
    max_buffer_steps=20, return_frames=True,
)
buffer = probe['recommended']
print(probe)
if buffer >= 0.01 * 20:
    raise RuntimeError('Probe reached its search limit; increase max_buffer_steps and rerun.')


## 5. MagMapSim

All named parameters are explicit, with one config keyword override:
`buffer_size=buffer`. Any other config field can be overridden the same way.
The result is saved regardless of `keep_in_memory`; a single-file input returns
`MagnificationMap`. Here the computed float32 array remains available directly.
One engine coordinates PMBuilder → P3MDeflector → IPM/ASR → rasterization.


In [ ]:
mmp = MagMapSim(
    lens=lens, config=config, output='map.npy', save_dir=run_dir / 'single',
    keep_in_memory=True, buffer_size=buffer,
)
print(mmp.path, mmp.metadata_path)
print(mmp.load().shape)
display(mmp.metadata['geometry'])


## 6. MagMapShow

All plotting parameters are shown. Metadata supplies `ps` and bounds. For a raw
array without metadata, pass the pixel scale explicitly. `cut` crops only the
view. `vmin`/`vmax` control logarithmic display, not map data. `show=False` returns
the figure for customization/saving.


In [ ]:
fig, ax = MagMapShow(
    source=mmp, ps=None, frame=0, cut=0, img_center=(0.0, 0.0),
    vmin=0.5, vmax=None, sys_name='Two-lens example', show=False,
)
fig.savefig(run_dir / 'magnification_map.png', dpi=150)
plt.show()

# Raw-array usage (all arguments shown):
fig_raw, ax_raw = MagMapShow(
    source=mmp.load(), ps=config.srcp_ps, frame=0, cut=0,
    img_center=config.img_center, vmin=0.5, vmax=None,
    sys_name='Raw array example', show=False,
)
plt.close(fig_raw)


## 7. IALightCurveExtractor and LightCurveResult

The constructor and `extract()` include every parameter. Endpoints are arcsec;
speed is arcsec/year; returned time uses days (365.25 days/year). Source radius
is pixels: `r=max(1,int(source_radius_px))`, sigma `r/2`, square support `[-r,r]²`.
Windows crossing the map boundary return magnification 1. No spatial interpolation
is added. `sample_dt_days=None` selects the established automatic density.

`max_vram_mb` is measured in MiB; the actual conservative budget also depends on
free GPU memory. Construction does not upload the map. `extract()` performs
batched GPU convolution of the required regions.


In [ ]:
extractor = IALightCurveExtractor(
    source=mmp, ps=None, frame_dt_days=None,
    pmv_arcsec_yr=0.005, max_vram_mb=256, img_center=(0.0, 0.0),
    title_mmp='Magnification Map', lc_title='Light Curve', gamma=0.1,
)
curve = extractor.extract(
    start=(-0.01, 0.0), end=(0.01, 0.0),
    source_radius_px=3, sample_dt_days=10.0,
)
curve_path = curve.save(path=run_dir / 'curve.npz')
fig, ax = plt.subplots()
ax.plot(curve.time_days, curve.magnification)
ax.set(xlabel='Time (days)', ylabel='Magnification', title='Finite-source light curve')
plt.show()
print(curve_path)


In [ ]:
# Explicit LightCurveResult construction from saved named arrays:
with np.load(curve_path, allow_pickle=False) as data:
    restored_curve = LightCurveResult(
        time_days=data['time_days'], magnification=data['magnification'],
        x=data['x'], y=data['y'], frame_index=data['frame_index'],
    )
print(restored_curve.time_days.shape)

# Explicit map construction: geometry is loaded from its JSON sidecar.
reloaded_map = MagnificationMap(
    array=None, path=mmp.path, metadata={}, metadata_path=mmp.metadata_path,
)
read_only_map = reloaded_map.load()
print(type(read_only_map), read_only_map.shape)
del read_only_map


## 8. Optional ensemble PSD example

Install `.[analysis]`, then set `RUN_PSD=True`. The legacy PSD interface expects
`fluxes` shaped `(n_samples,T)` and `times` in years with a **365-day convention**.
This example selects the uniformly spaced part of the extracted curve, omitting
any extra shorter endpoint interval, then converts days explicitly by dividing
by 365.0. One curve demonstrates the interface; scientific ensemble inference
requires an appropriate ensemble of independent curves.

All function arguments and all used config keys are explicit. `constraints` here
means minimum N; the observation analysis below instead uses maximum N.


In [ ]:
RUN_PSD = False
if RUN_PSD:
    from DCLens.analysis import power_spectrum as psd
    uniform = np.isclose(curve.time_days / 10.0,
                         np.round(curve.time_days / 10.0))
    fluxes = curve.magnification[uniform][None, :]
    times = curve.time_days[uniform] / 365.0
    info = psd.compute_group_psd_matrix(
        fluxes=fluxes, times=times, psd_mode='fft', nperseg_default=256,
        n_freq_bins=120, welch_window='hann', welch_detrend='linear',
    )
    psd_dir = run_dir / 'psd'
    psd_dir.mkdir(exist_ok=True)
    name = psd.combo_name(m=1, N=2, rh=1)
    ensemble_path = psd_dir / f'LC_{name}_R3.npz'
    np.savez(ensemble_path, fluxes=fluxes, times=times)
    psd_cfg = dict(
        out_dir=str(psd_dir), m_list=[1], N_list=[2], rh_list=[1],
        constraints={(1, 1): 2}, source_size_pixels=3,
        psd_mode='fft', nperseg_default=256, n_freq_bins=120,
        welch_window='hann', welch_detrend='linear',
        figsize=(10, 6), xscale='log', markevery=10, markersize=4,
        line_width=1.5, line_alpha=0.8, label_fontsize=12,
        legend_fontsize=10, legend_title_fontsize=11,
        output_fig_name='groups.png', dpi=150, n_single_samples=3,
    )
    combinations = list(psd.iter_valid_combinations(cfg=psd_cfg))
    selected_path = psd.resolve_latest_npz(
        out_dir=psd_dir, combo_name_str=name, r_px=3,
    )
    groups = psd.plot_psd_by_params(cfg=psd_cfg, single_samples=False)
    single_info = psd.plot_single_npz_psd_with_1sigma(
        npz_path=selected_path,
        cfg=dict(psd_mode='fft', nperseg_default=256, n_freq_bins=120,
                 welch_window='hann', welch_detrend='linear', figsize=(8, 6),
                 xscale='log', label_fontsize=12, title='Example PSD',
                 title_fontsize=14, legend_fontsize=10,
                 out_dir=str(psd_dir), output_fig_name='single.png', dpi=300),
    )


## 9. Optional observation matching

The cell below is a template for **your own observation CSV** with columns `mjd`,
`band`, `mag`, `magerr`. Supply the path and enable the switch only when ready.
The map is copied into the analysis module's existing filename convention.
All configuration keys are shown. Full-map CPU convolution here uses the legacy
four-sigma source model, which is different from the interactive extractor.
Use a duration and proper motion whose horizontal track fits this small map.
The appendix includes the complete lower-level observation function workflow.


In [ ]:
RUN_OBSERVATION = False
if RUN_OBSERVATION:
    from DCLens.analysis.observation import OBSandSIM_Match
    observation_csv = Path('observations.csv')  # Replace with your data.
    observation_maps = run_dir / 'observation_maps'
    observation_maps.mkdir(exist_ok=True)
    np.save(observation_maps / 'm_1_N2_rh_1.npy', mmp.load(), allow_pickle=False)
    observation_cfg = dict(
        obs_csv=str(observation_csv), base_dir=str(observation_maps),
        output_dir=str(run_dir / 'observation_results'), output_csv='matches.csv',
        m_list=[1], rh_list=[1], N_list=[2], constraints={(1, 1): 2},
        source_size_pixels=3, proper_motion=0.005, pixel_scale=config.srcp_ps,
        random_seed=42, n_paths=100, filter_mode='relative', signal_threshold=0.05,
        replace_small_values=False, small_value_threshold=0.0, save_plots=True,
    )
    OBSandSIM_Match(cfg=observation_cfg)


## 10. Logging

`configure(verbose=False)` has one parameter. Probing and simulation already call
it using the config's `verbose`. `show_progress` separately controls simulation
progress printing. No log file is created.


In [ ]:
from DCLens.logger import configure
configure(verbose=False)


# Complete API and parameter reference

The following reference is included directly in the notebook so it remains
self-contained when shared. Defaults describe the library, not the smaller
example settings above. Code blocks in this appendix are usage templates, not
additional automatically executed notebook cells. They include the lower-level
observation workflow and legacy aliases. The `cfg`, `cfg_psd`, `fluxes`, `times`
and `mmp` variables in those standalone snippets refer to the inputs described
in their sections.


## Public API reference

All arguments below are listed, including defaults. `*` marks keyword-only
arguments. Methods with `self` use the instance and are shown without `self`.


### Lens

```python
from DCLens import Lens
Lens(source, D_L_pc=1e4, D_S_pc=1e9, N_planes=1, pattern='*.dat')
lens.load_frame(i)
lens.iter_frames()
```

| Parameter | Meaning |
| --- | --- |
| `source` | File path, directory path, or iterable of file paths. A directory uses natural filename order (`state2` before `state10`); an iterable preserves supplied order. Even a one-element iterable is a sequence. |
| `D_L_pc=1e4` | Reference lens distance in pc. |
| `D_S_pc=1e9` | Source distance in pc; finite `0 < D_L_pc < D_S_pc` is required. |
| `N_planes=1` | Positive integer plane count; splitting uses at most the number of retained lenses. |
| `pattern='*.dat'` | Glob pattern used only for directory input. |
| `i` | Zero-based snapshot index for `load_frame`; normal Python negative indexing also applies. |

Construction checks paths but does not load their tables. `load_frame(i)` returns
a pandas DataFrame with `x_pc`, `y_pc`, `M_Msun`, absolute `z_pc` and positive-distance
lenses. `iter_frames()` yields those tables lazily. `frame_count` is the number of
files; `files` is a tuple of absolute paths; `mode` is `'single'` or `'sequence'`.
These methods take no other parameters.


### SimulationConfig

```python
from DCLens import SimulationConfig
config = SimulationConfig(
    coarse_ps=1e-3, finest_ps=1e-4, distortion_threshold=0.2,
    PM_grid=2048, img_center=(0.0, 0.0), src_considered_region=(0.1, 0.1),
    buffer_size=0.1, max_refine_iters=20, srcp_ps=1e-4, ss=4,
    max_coarse_per_chunk=200, max_active_cells=20_000_000,
    p3m_a_pm_over_H=6.0, p3m_padding_factor=2.0,
    show_progress=True, verbose=False, deflector='p3m',
)
```

| Parameter | Default | Meaning |
| --- | --- | --- |
| `coarse_ps` | `1e-3` | Positive initial image-plane cell size, arcsec. |
| `finest_ps` | `1e-4` | Positive refinement stopping scale, arcsec. |
| `distortion_threshold` | `0.2` | Nonnegative triangle-area imbalance threshold. |
| `PM_grid` | `2048` | Positive integer active PM resolution on each axis, before padding. |
| `img_center` | `(0.0, 0.0)` | Finite `(x,y)` center in arcsec, used for image and source regions. |
| `src_considered_region` | `(0.1, 0.1)` | Positive `(width,height)` source region in arcsec. |
| `buffer_size` | `0.1` | Nonnegative image-plane margin added on each side, arcsec. |
| `max_refine_iters` | `20` | Positive integer maximum ASR iterations per chunk; unfinished refinement raises. |
| `srcp_ps` | `1e-4` | Positive source-map pixel size, arcsec/pixel. |
| `ss` | `4` | Positive integer raster supersampling factor per axis. |
| `max_coarse_per_chunk` | `200` | Positive integer coarse cells per chunk axis. |
| `max_active_cells` | `20_000_000` | Positive capacity of each active queue and result queue; at most `(2**31-1)//4`. |
| `p3m_a_pm_over_H` | `6.0` | Positive P3M splitting radius divided by mesh spacing. |
| `p3m_padding_factor` | `2.0` | Positive FFT-domain padding factor. |
| `show_progress` | `True` | Print chunk and output progress during simulation. |
| `verbose` | `False` | Enable detailed package/native diagnostics. |
| `deflector` | `'p3m'` | `'p3m'` or `'pp'`; PP sums all lenses per plane without PM/bucket setup. |

The dataclass is frozen. Use `dataclasses.replace(config, buffer_size=value)` for
a modified copy, or pass field overrides directly to the two functions below.
Overrides take precedence over `config` without mutating it. Unknown keywords are
rejected. PM settings are unused in PP mode but still undergo config validation.
The **20-million** queue capacity is the Python default; it alone consumes about
3.28 GiB when allocated. Small examples explicitly reduce it.


### BufferSizeProbe

```python
from DCLens import BufferSizeProbe
BufferSizeProbe(lens, *, config=None, buffer_step=0.01,
                max_buffer_steps=200, return_frames=False, **keywords)
```

| Parameter | Meaning |
| --- | --- |
| `lens` | A `Lens` instance. All snapshots are probed. |
| `config=None` | `SimulationConfig`; `None` constructs the defaults. |
| `buffer_step=0.01` | Positive candidate-margin increment in arcsec. |
| `max_buffer_steps=200` | Positive integer number of candidate margins. |
| `return_frames=False` | False returns a scalar maximum margin; true returns `{'recommended': scalar, 'frames': list}` in snapshot order. |
| `**keywords` | Any of the 17 `SimulationConfig` fields above, with exactly those defaults when no config is provided. |

The probe prepares the same selected deflector as simulation, but does not allocate
IPM queues or create a rasterizer. It searches with image bounds based on
`max_buffer_steps * buffer_step`; the configured `buffer_size` does not determine
this search range. Refinement, source pixel size, chunk size, supersampling and
simulation progress controls do not drive the boundary probe. The recommended
margin must be passed explicitly to `MagMapSim`; the input config is not modified.


### MagMapSim

```python
from DCLens import MagMapSim
MagMapSim(lens, *, config=None, output=None, save_dir=None,
          keep_in_memory=None, **keywords)
```

| Parameter | Meaning |
| --- | --- |
| `lens` | `Lens` instance, single or sequence. |
| `config=None` | `SimulationConfig`, or defaults when omitted. |
| `output=None` | Single-map filename/path; default `map.npy`. A missing `.npy` suffix is appended. Relative paths resolve under `save_dir`; an absolute path takes precedence. Must be `None` for sequences. |
| `save_dir=None` | Output directory; default current directory for single maps and `maps` for sequences. Created if necessary. |
| `keep_in_memory=None` | Single maps default to true, sequences to false. True is rejected for sequences. False keeps disk paths/metadata only. |
| `**keywords` | Any of the 17 config fields; per-call overrides take precedence. |

Returns `MagnificationMap` for a single file, or `MagnificationMapSeries` for a
directory/list input. Every map is saved even when retained in memory. One native
engine is reused through the run and closed in `finally`, including on failure.
Use the same region, lens geometry and deflector configuration for probing and
simulation. PP changes only deflection setup/evaluation; IPM, ASR and rasterization
retain their usual path.


### PMBuilder and lens-plane preparation

```python
from DCLens import PMBuilder
from DCLens.physics import split_into_planes, apply_p3m_grid
planes = split_into_planes(BH_table, N_planes)
planes = PMBuilder(lens_planes=planes, grid_nx=128, grid_ny=128,
                   theta_bounds_arcsec=None, a_pm_over_H=6.0,
                   padding_factor=2.0)
```

`split_into_planes(BH_table, N_planes)` takes a DataFrame with `x_pc`, `y_pc`,
`M_Msun`, **absolute** `z_pc`, and the requested integer number of planes. Use
`lens.load_frame(i)` as the table. It returns distance-ordered dictionaries,
with mean `z_pc` and `lenses` arrays (`x_pc`, `y_pc`, `M_Msun`), partitioned into
nearly equal lens counts. Empty input returns an empty list.

| PMBuilder parameter | Meaning |
| --- | --- |
| `lens_planes` | Plane list returned by `split_into_planes`. Modified in place and returned. |
| `grid_nx=128`, `grid_ny=128` | Positive active mesh resolutions; these low-level defaults differ from `SimulationConfig.PM_grid`. |
| `theta_bounds_arcsec=None` | Optional `(xmin,xmax,ymin,ymax)` image-ray footprint, arcsec; `None` uses lens extents. |
| `a_pm_over_H=6.0` | Positive cutoff/spacing ratio. |
| `padding_factor=2.0` | Positive FFT-box padding factor. |

The builder adds `p3m` dictionaries and stably reorders lens arrays by PP bucket.
Each dictionary contains `pm_nx`, `pm_ny`, `pm_xmin`, `pm_ymin`, `H`, `a_pm`,
`raw_nx`, `raw_ny`, `active_grid_nx`, `active_grid_ny`, `pp_nx`, `pp_ny`,
`pp_xmin`, `pp_ymin`, `pp_h`, `bucket_start`, `bucket_count`. Physical mesh
coordinates/spacing are in pc; bucket starts/counts are int32 arrays.
`PMBuilder` and `apply_p3m_grid` are the same function with the same signature.
Simulation calls this preparation automatically; advanced inspection does not
replace the normal `MagMapSim` input, which remains a `Lens`.


### MagnificationMap and MagnificationMapSeries

```python
from DCLens import MagnificationMap, MagnificationMapSeries
MagnificationMap(array=None, path=None, metadata={}, metadata_path=None)
MagnificationMapSeries(manifest)
mmp.load()
series.frame(i)
series.validate()
```

The displayed `metadata={}` represents a **fresh** dict per instance.

| Parameter | Meaning |
| --- | --- |
| `array=None` | Existing 2-D numeric NumPy map, retained without copying. |
| `path=None` | `.npy` path for disk access. At least `array` or `path` is required. |
| `metadata={}` | Map metadata dict; if empty, load the resolved metadata sidecar when available. |
| `metadata_path=None` | Explicit JSON sidecar path; otherwise discover same-stem `.json` beside `path`. |
| `manifest` | Version-1 sequence manifest path or directory containing `manifest.json`. |
| `i` | Frame index; normal Python indexing. |

`mmp.load()` returns the supplied array or a read-only NumPy memmap. It does not
cache a new in-memory copy. `series.frame(i)` returns a lazy `MagnificationMap`.
`series.frame_count`, `frames`, `geometry`, `simulation`, and `manifest_path`
expose manifest records. `series.validate()` checks all headers/sidecars and
common geometry without intentionally loading all pixels, returning `None`.
The loader methods have no other parameters. For arrays without metadata, give
`ps` and optionally `img_center` to display/extraction.

```python
mmp = MagnificationMap(array=None, path='results/map.npy',
                       metadata={}, metadata_path='results/map.json')
array = mmp.load()
series = MagnificationMapSeries(manifest='dynamic_maps/manifest.json')
first = series.frame(i=0)
```


### MagMapShow

```python
from DCLens import MagMapShow
MagMapShow(source, ps=None, *, frame=0, cut=0, img_center=(0.0, 0.0),
           vmin=0.5, vmax=None, sys_name='N-body Lens System', show=True)
```

| Parameter | Meaning |
| --- | --- |
| `source` | `MagnificationMap`, `MagnificationMapSeries`, 2-D NumPy array, `.npy` path, manifest path, or sequence directory. |
| `ps=None` | Arcsec/pixel. Required for an array/NPY without geometry metadata; if supplied with metadata it must agree. |
| `frame=0` | Frame index when the source is a series. |
| `cut=0` | Nonnegative integer pixels cropped from each edge for display only; must leave a nonempty map. |
| `img_center=(0.0,0.0)` | Arcsec center for geometry inferred without metadata. |
| `vmin=0.5` | Positive logarithmic lower limit; `None` also selects `0.5`. |
| `vmax=None` | Upper limit; default `max(shown.max(), vmin*1.01)`; must exceed lower limit. |
| `sys_name='N-body Lens System'` | Name inserted into the map title. |
| `show=True` | Call Matplotlib `show`; false allows saving/customizing first. |

Returns `(figure, axes)`, using `magma` and logarithmic normalization.
Example: `fig, ax = MagMapShow(mmp, show=False); fig.savefig('map.png')`.


### IALightCurveExtractor

```python
from DCLens import IALightCurveExtractor
IALightCurveExtractor(source, ps=None, *, frame_dt_days=None,
                     pmv_arcsec_yr=0.005, max_vram_mb=2048,
                     img_center=(0.0, 0.0), title_mmp='Magnification Map',
                     lc_title='Light Curve', gamma=0.1)
extractor.extract(start, end, source_radius_px=5, *, sample_dt_days=None)
extractor.show()
```

| Constructor parameter | Meaning |
| --- | --- |
| `source` | Same source types as `MagMapShow`; a series uses all snapshots. |
| `ps=None` | Arcsec/pixel; required without metadata, must agree with metadata otherwise. |
| `frame_dt_days=None` | Positive uniform snapshot duration in days, required for a sequence; unused for a single map. |
| `pmv_arcsec_yr=0.005` | Positive angular speed along the source path, arcsec/year. |
| `max_vram_mb=2048` | Positive requested extraction budget in MiB (despite the historical parameter name); `None` uses the same 2048 MiB cap. |
| `img_center=(0.0,0.0)` | Center in arcsec when geometry has no metadata. |
| `title_mmp='Magnification Map'` | Interactive map title. |
| `lc_title='Light Curve'` | Interactive curve title. |
| `gamma=0.1` | PowerNorm gamma for the interactive map display; does not change convolution. |

| `extract` parameter | Meaning |
| --- | --- |
| `start`, `end` | Finite `(x,y)` path endpoints in arcsec. |
| `source_radius_px=5` | Finite source size in pixels; uses `r=max(1,int(value))`. |
| `sample_dt_days=None` | Positive sampling interval in days; `None` uses approximately one sample per map pixel along the full path, at least ten for a nonzero full path. |

`extract()` returns `LightCurveResult`. The normalized discrete Gaussian has
`sigma=r/2` and square support `[-r,r]²`. Window centers use integer pixel
coordinates with the existing truncation convention, without interpolation.
Any window not entirely inside the map returns magnification `1.0`.

Travel duration is angular path length / proper motion × **365.25** days. Frame
`i` covers `[i*frame_dt_days,(i+1)*frame_dt_days)` with no temporal interpolation.
Sampling ends when the path ends or the sequence runs out; the final sequence
right boundary is excluded. A stationary path produces one sample at time zero.

Construction opens geometry/header data but does not plot or upload maps.
`show()` takes no arguments and returns `(figure,(map_axes,curve_axes))`. Click two
map positions for the start/end; move the radius slider (1–100 px) to update the
curve. The preview stays on frame zero even for dynamic extraction. Interactive
extraction uses the default `sample_dt_days=None`. For notebook interaction:

```python
%matplotlib widget
extractor.show()
```

Install the notebook extra first. Use `%matplotlib inline` for static figures;
it does not support the two-click interface. The UI uses a bounded preview while
convolution always uses the original map resolution.


### LightCurveResult

```python
from DCLens import LightCurveResult
LightCurveResult(time_days, magnification, x, y, frame_index)
curve.save(path)
```

The five required constructor arguments are aligned 1-D arrays: time in days,
dimensionless magnification, continuous x/y arcsec positions, and zero-based
snapshot indices (all zero for a single map). Normally use the object returned
by `extract()`. `save(path)` accepts a path, appends `.npz` if absent, creates
parents, writes the five named arrays and returns the resulting `Path`.
Existing output may be overwritten.

```python
import numpy as np
with np.load('results/curve.npz', allow_pickle=False) as data:
    time_days = data['time_days']
    magnification = data['magnification']
```


### Logging

`DCLens.logger.configure(verbose=False)` configures the package console logger;
`verbose=True` selects DEBUG, otherwise WARNING. It returns `None` and creates
no log file. Simulation/probing call it automatically. `show_progress` is a
separate simulation print control.

## Optional analysis API

Install `.[analysis]`. These modules preserve their separate scientific
conventions and are not implicitly imported by `import DCLens`. All module-defined
public functions are documented below; imported third-party functions are not
DCLens APIs. Analysis configuration dictionaries have no implicit defaults except
where explicitly listed.


### Observation matching

```python
from DCLens.analysis.observation import OBSandSIM_Match
cfg = dict(
    obs_csv='observations.csv', base_dir='observation_maps',
    output_dir='observation_results', output_csv='matches.csv',
    m_list=[1], rh_list=[1], N_list=[100], constraints={(1, 1): 100},
    source_size_pixels=5, proper_motion=0.005, pixel_scale=0.0005,
    random_seed=42, n_paths=100, filter_mode='relative', signal_threshold=0.05,
    replace_small_values=False, small_value_threshold=0.0, save_plots=True,
)
OBSandSIM_Match(cfg=cfg)
```

| `cfg` key | Meaning |
| --- | --- |
| `obs_csv` | CSV with `mjd`, `band`, `mag`, `magerr`; uses the most populated band. |
| `base_dir` | Directory containing `m_{m}_N{N}_rh_{rh}.npy` maps. |
| `output_dir`, `output_csv` | Directory and summary CSV filename. |
| `m_list`, `rh_list`, `N_list` | Parameter labels to scan; formatting must match filenames. |
| `constraints` | Mapping `(m,rh)` to **maximum** allowed N; absent combinations are skipped. |
| `source_size_pixels` | Gaussian radius parameter in px, sigma `int(radius)/2`, support to four sigma. |
| `proper_motion` | Angular motion, arcsec/year; uses 365.25 days/year. |
| `pixel_scale` | Map scale, arcsec/pixel. |
| `random_seed` | Seed accepted by NumPy's default RNG. |
| `n_paths` | Number of random horizontal paths attempted. |
| `filter_mode` | `'absolute'` for absolute peak-to-peak filtering; any other value uses peak-to-peak divided by mean. |
| `signal_threshold` | Minimum variation passing the chosen filter. |
| `replace_small_values` | Replace pixels at/below the threshold with 1 before convolution. |
| `small_value_threshold` | Replacement threshold, used when replacement is enabled. |
| `save_plots` | Save observation/PSD comparison images. |

`OBSandSIM_Match(cfg)` returns `None`; writes a summary only if results exist,
and optional `{map_name}_match.png` images. It retains full-map CPU FFT convolution,
noise addition, Lomb–Scargle power and the existing candidate criteria. It does not
consume `LightCurveResult` files or sequence manifests directly.

All lower-level functions in `DCLens.analysis.observation` are also callable:

| Function (complete signature) | Parameters and return |
| --- | --- |
| `make_pure_gaussian_kernel(r_px)` | Pixel radius; returns `(normalized_kernel, support_radius)`. `int(r_px)<1` returns a 1×1 unit kernel and radius 1. |
| `process_one_map(map_arr, kernel, r_px)` | 2-D map, normalized 2-D kernel, crop/support radius in px; returns `(convolved_map, valid_x_indices, valid_y_indices)`. |
| `is_signal_significant(flux, cfg)` | 1-D flux; dict with `filter_mode`, `signal_threshold`; returns a boolean. |
| `prepare_observation(csv_path)` | CSV path as above; returns `(relative_days, normalized_flux, flux_error, selected_band)`. |
| `extract_obs_matched_curves(conv_map, valid_x, valid_y, t_rel, flux_err_obs, cfg)` | Convolved 2-D map, valid integer coordinate arrays, observation relative days/errors; cfg needs `proper_motion`, `pixel_scale`, `random_seed`, `n_paths`, `filter_mode`, `signal_threshold`. Returns accepted noisy flux curves; no acceptance yields an empty array. |
| `calculate_criteria(f_obs, sim_fluxes, freqs, psd_obs, psd_sims)` | Observation flux `(T,)`, simulated flux `(n,T)`, frequencies `(F,)` in inverse days, observation power `(F,)`, simulated powers `(n,F)`; returns `(red_chi2, overlap_percent, obs_std, sim_mean_std, pull, median_psd, p16, p84, candidate_string)`. |
| `plot_and_save(t_obs, f_obs, f_err, sim_fluxes, freqs, psd_obs, median_psd, p16, p84, pull, map_name, save_dir)` | Relative days, flux/errors, simulated curves, frequencies and PSD summary arrays from the preceding functions, output stem and existing directory; saves `{map_name}_match.png`, returns `None`. |

A direct lower-level workflow (all parameters shown):

```python
import numpy as np
from astropy.timeseries import LombScargle
from pathlib import Path
from DCLens.analysis import observation as obs

t, f, err, band = obs.prepare_observation(csv_path=cfg['obs_csv'])
kernel, radius = obs.make_pure_gaussian_kernel(r_px=cfg['source_size_pixels'])
conv, vx, vy = obs.process_one_map(map_arr=mmp.load(), kernel=kernel, r_px=radius)
curves = obs.extract_obs_matched_curves(conv_map=conv, valid_x=vx, valid_y=vy,
                                       t_rel=t, flux_err_obs=err, cfg=cfg)
if len(curves):
    significant = obs.is_signal_significant(flux=curves[0], cfg=cfg)
    freqs = np.logspace(np.log10(1/(t.max()+1)),
                        np.log10(0.5/np.median(np.diff(t))), 150)
    p_obs = LombScargle(t, f).power(freqs)
    p_sims = np.array([LombScargle(t, row).power(freqs) for row in curves])
    criteria = obs.calculate_criteria(f_obs=f, sim_fluxes=curves, freqs=freqs,
                                      psd_obs=p_obs, psd_sims=p_sims)
    _, _, _, _, pull, median, p16, p84, _ = criteria
    Path(cfg['output_dir']).mkdir(parents=True, exist_ok=True)
    obs.plot_and_save(t_obs=t, f_obs=f, f_err=err, sim_fluxes=curves,
                      freqs=freqs, psd_obs=p_obs, median_psd=median,
                      p16=p16, p84=p84, pull=pull, map_name='example',
                      save_dir=cfg['output_dir'])
```

Use an observation cadence and duration that fit within the map's available
horizontal path. Real observations should contain enough distinct times for the
retained frequency construction. The four-sigma source model here differs from
the interactive extractor's truncated square; the two are not silently unified.


### Ensemble power spectra

```python
from DCLens.analysis.power_spectrum import compute_group_psd_matrix
info = compute_group_psd_matrix(
    fluxes=fluxes, times=times, psd_mode='fft', nperseg_default=256,
    n_freq_bins=120, welch_window='hann', welch_detrend='linear',
)
```

| Parameter | Default | Meaning |
| --- | --- | --- |
| `fluxes` | required | Nonempty real ensemble `(n_samples,T)`. |
| `times` | required | Common uniformly spaced time array `(T,)` in **years**, with enough samples for distinct positive frequency bins; the routine derives cadence from the first interval. |
| `psd_mode` | `'fft'` | `'fft'` periodogram or `'welch'`. |
| `nperseg_default` | `256` | Welch segment length, capped at T. |
| `n_freq_bins` | `120` | Number of common logarithmic frequency bins. |
| `welch_window` | `'hann'` | SciPy Welch window specification. |
| `welch_detrend` | `'linear'` | SciPy Welch detrending specification. |

Returns a dict with `freq_grid` (1/day), `X_log` (log10 PSD for each curve),
`median_log10`, `dt_days`, `fs` (1/day), and `n_samples`. This existing routine
uses **365 days/year**, unlike the trajectory's 365.25. To preserve day spacing
when adapting a uniformly sampled `LightCurveResult`, explicitly set
`times=curve.time_days/365.0`, `fluxes=curve.magnification[None,:]`. Do not pass a
nonuniform final endpoint to this uniformly sampled PSD routine; select/resample
a uniform segment first. Native ensemble files have arrays `fluxes` and `times`;
they are not the same format as `LightCurveResult.save()`.

```python
from DCLens.analysis import power_spectrum as psd
cfg_psd = dict(
    out_dir='psd_results', m_list=[1], N_list=[100], rh_list=[1],
    constraints={(1, 1): 100}, source_size_pixels=5,
    psd_mode='fft', nperseg_default=256, n_freq_bins=120,
    welch_window='hann', welch_detrend='linear',
    figsize=(10, 6), xscale='log', markevery=10, markersize=4,
    line_width=1.5, line_alpha=0.8, label_fontsize=12,
    legend_fontsize=10, legend_title_fontsize=11,
    output_fig_name='psd_groups.png', dpi=150, n_single_samples=3,
)
groups = psd.plot_psd_by_params(cfg=cfg_psd, single_samples=False)
```

`plot_psd_by_params(cfg, single_samples=False)` reads
`LC_m{m}_N{N}_rh{rh}_R{source_size_pixels}.npz` in `out_dir`, writes per-group
`psd_median_{combo}_{mode}.csv`, a `psd_summary_{mode}.csv`, and the configured
figure, shows the plot and returns a dict keyed by combo name. Missing files are
skipped; no available groups raises. Configuration keys:

| Keys | Meaning / defaults |
| --- | --- |
| `out_dir` | Input ensemble directory and output directory, required. |
| `m_list`, `N_list`, `rh_list` | Parameter labels and plotting legend order, required. |
| `constraints` | `(m,rh)` → **minimum** N allowed, required; this is intentionally opposite to observation matching. |
| `source_size_pixels` | Radius token in input filenames, required. |
| `psd_mode`, `nperseg_default`, `n_freq_bins` | Required; semantics as `compute_group_psd_matrix`. |
| `welch_window`, `welch_detrend` | Optional, defaults `'hann'`, `'linear'`. |
| `figsize` | Figure `(width,height)` in inches, required. |
| `xscale` | `'log'` for logarithmic frequency axis, otherwise linear, required. |
| `markevery`, `markersize` | Matplotlib marker spacing and size, required. |
| `line_width`, `line_alpha` | Median line width and opacity, required. |
| `label_fontsize`, `legend_fontsize`, `legend_title_fontsize` | Font sizes, required. |
| `output_fig_name`, `dpi` | Output figure name and resolution, required. |
| `n_single_samples` | Optional number of randomly drawn individual spectra when `single_samples=True`; default 3. |

Single-file plotting:

```python
info = psd.plot_single_npz_psd_with_1sigma(
    npz_path='psd_results/LC_m1_N100_rh1_R5.npz',
    cfg=dict(psd_mode='fft', nperseg_default=256, n_freq_bins=120,
             welch_window='hann', welch_detrend='linear', figsize=(8, 6),
             xscale='log', label_fontsize=12, title='PSD example',
             title_fontsize=14, legend_fontsize=10,
             out_dir='psd_results', output_fig_name='single.png', dpi=300),
)
```

`plot_single_npz_psd_with_1sigma(npz_path, cfg)` returns the PSD dict, shows median,
16th/84th percentile band and up to three random members. All its cfg keys are
optional: defaults are the values shown above, except `title` defaults to
`PSD Analysis for {filename}`, and no image is saved unless **both** `out_dir`
and `output_fig_name` are present. `dpi=300` applies when saving. It does not use
the batch-only parameter lists, constraints, styles or `n_single_samples`.

Additional complete helper signatures:

| Function | Parameters and return |
| --- | --- |
| `iter_valid_combinations(cfg)` | Dict containing `m_list`, `rh_list`, `N_list`, `constraints`; yields `(m,N,rh)` with `N >= constraints[(m,rh)]`. |
| `combo_name(m, N, rh)` | Three parameter labels; returns `m{m}_N{N}_rh{rh}`. |
| `resolve_latest_npz(out_dir, combo_name_str, r_px=5)` | Directory, combo name string, radius filename token; returns the newest matching `LC_{combo}_R{r_px}.npz` path or `None`. |

```python
combinations = list(psd.iter_valid_combinations(cfg=cfg_psd))
name = psd.combo_name(m=1, N=100, rh=1)
path = psd.resolve_latest_npz(out_dir=cfg_psd['out_dir'],
                              combo_name_str=name, r_px=5)
```

## Resources and troubleshooting

- **Memory:** the Python default `max_active_cells=20_000_000` reserves two
  48-byte cell queues and one 80-byte leaf queue, about 3.28 GiB, only when IPM
  runs. Start with smaller explicit capacities/chunks on constrained GPUs.
  Raster targets scale as `nx * ny * ss²`; PM FFT dimensions also grow with
  padding and power-of-two rounding. Default settings are not guaranteed to
  fit every GPU.
- **Queue capacity or unfinished refinement:** reduce `max_coarse_per_chunk`,
  or select an appropriate larger `max_active_cells` / `max_refine_iters`.
  These choices retain the existing numerical implementation but can affect
  resource use; scientific resolution settings should be chosen deliberately.
- **Extraction:** buffers use a budget of
  `0.8 * min(requested MiB, 0.5 * currently free GPU bytes)` after unit conversion.
  Maps are read-only mmap when disk-backed; only required path regions are
  uploaded. Batches shrink on memory pressure, and failure to fit one source
  window raises. Whole maps are not retained for a sequence.
- **Import/build errors:** use the Python environment in which DCLens was
  installed; rebuild after native changes. Ensure `nvcc`, CUDA headers/libraries,
  Python headers, and OpenGL/EGL development files are discoverable. Change
  `CMAKE_CUDA_ARCHITECTURES` when the default SM 86 is inappropriate. Stale local
  extensions can shadow a separately installed package when running in a checkout.
- **EGL/interoperability errors:** the renderer requires NVIDIA-backed OpenGL/EGL
  and access to the matching CUDA device, including inside containers. A software
  OpenGL implementation is insufficient for this GPU interop path.
- **Output directory exists:** select a fresh directory for sequence simulation.
  The simulator does not resume a partial sequence automatically.
- **No interactive clicks:** enable `%matplotlib widget` with `ipympl`, or use
  a desktop interactive Matplotlib backend. Programmatic `extract()` needs no GUI.
- **Scientific conventions:** TSC assignment, P3M force splitting, plane
  propagation, IPM area/refinement criteria, short-diagonal triangulation,
  additive R32F rendering, supersampling and float32 chunk accumulation retain
  their existing implementations. The buffer heuristic, full coarse-cell extent
  at partial edge chunks, and separate analysis source/time conventions are
  retained as well.

The Quick Start notebook is distributed without precomputed outputs.
